# SAM augmentations and VQA-RAD preparation

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the loc_lens repository."
sys.path.insert(0, str(ROOT / "src"))

MANIFEST = ROOT / "data/vqarad/manifest.jsonl"
LENSES = ROOT / "data/vqarad/lenses"

In [2]:
import subprocess

def run_module(module, arguments):
    subprocess.run(
        [sys.executable, "-u", "-m", f"localization_lens.{module}", *map(str, arguments)],
        cwd=ROOT,
        check=True,
    )


## Configuration
Keep the compact defaults for a demo. Set `PREPARE_DATA = True` only when you intend to prepare/rewrite the manifest. Leave it false to reuse your existing dataset.

In [3]:
PREPARE_DATA = True
MAX_TRAIN = 1_000_000
MAX_TEST = 1_000_000
MAX_IMAGES = 1_000_000

SAM_MODEL = "facebook/sam-vit-huge"
POINTS_PER_BATCH = 64
POINTS_PER_CROP = 64
CROPS_N_LAYERS = 1
PRED_IOU_THRESH = 0.88
STABILITY_SCORE_THRESH = 0.95
OVERWRITE = True
DEVICE = "cuda"

In [4]:
if PREPARE_DATA:
    run_module("data", ["--output", MANIFEST.parent, "--max-train", MAX_TRAIN, "--max-test", MAX_TEST])
else:
    assert MANIFEST.exists(), "No manifest found. Set PREPARE_DATA=True and run this cell again."


Wrote 2244 QA rows to /home/hasan/extras/loc_lens/data/vqarad/manifest.jsonl


## Generate SAM views
This may download the SAM weights on its first run. Lower `POINTS_PER_BATCH` if GPU memory runs out.

In [5]:
run_module("sam_augment", [
    "--manifest", MANIFEST, "--output", LENSES,
    "--model", SAM_MODEL, "--device", DEVICE,
    "--max-images", MAX_IMAGES, "--points-per-batch", POINTS_PER_BATCH,
    "--pred-iou-thresh", PRED_IOU_THRESH,
    "--points-per-crop", POINTS_PER_CROP,
    "--crops-n-layers", CROPS_N_LAYERS,
    "--stability-score-thresh", STABILITY_SCORE_THRESH,
    *(["--overwrite"] if OVERWRITE else []),
])


Loading weights: 100%|██████████| 594/594 [00:00<00:00, 25701.66it/s]
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


skip 7937c2ca96b19300: No usable masks after area filtering
skip e0a641fd1a925753: No usable masks after area filtering
skip 32e2af084cb94531: No usable masks after area filtering
skip f58f84f3dbd318bd: No usable masks after area filtering
skip 154e37b068c21374: No usable masks after area filtering
skip 445506f4ec41905c: No usable masks after area filtering
skip bc460819a996278d: No usable masks after area filtering
skip 0dac7e9da2692fe5: No usable masks after area filtering
skip 8b1aefbb99a07dcc: No usable masks after area filtering
skip fe7463bfd6df3d32: No usable masks after area filtering
skip 07538d1717a57650: No usable masks after area filtering
skip 006ae70657341fcf: No usable masks after area filtering
skip dafe9709c8ac31f3: No usable masks after area filtering
skip bfa1a4e97fbcd58b: No usable masks after area filtering
skip cdde5f4a7a42680b: No usable masks after area filtering
skip ad86a64acf54b430: No usable masks after area filtering
SAM views ready for 298/314 unique image